In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**FD001-FD004**

In [17]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "/content/drive/MyDrive/TASK-ERAU/C-MAPSS"
FD = "FD004"
DATA_DIR = f"{BASE_DIR}/{FD}"

train_path = f"{DATA_DIR}/train_{FD}.txt"
test_path  = f"{DATA_DIR}/test_{FD}.txt"
rul_path   = f"{DATA_DIR}/RUL_{FD}.txt"

print("Train exists:", os.path.exists(train_path))
print("Test exists:", os.path.exists(test_path))
print("RUL exists:", os.path.exists(rul_path))

columns = (
    ["unit", "cycle"]
    + ["setting_1", "setting_2", "setting_3"]
    + [f"s{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(train_path, sep=r"\s+", header=None, names=columns)
test_df  = pd.read_csv(test_path, sep=r"\s+", header=None, names=columns)
rul_df   = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["RUL"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)

display(train_df.head())

Train exists: True
Test exists: True
RUL exists: True
Train shape: (61249, 26)
Test shape: (41214, 26)
RUL shape: (248, 1)


,unit,cycle,setting_1,setting_2,setting_3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754


In [18]:
FAULT_THRESHOLD = 30
RUL_CAP = 125

# Training RUL
train_df["max_cycle"] = train_df.groupby("unit")["cycle"].transform("max")
train_df["RUL_raw"] = train_df["max_cycle"] - train_df["cycle"]
train_df["RUL"] = train_df["RUL_raw"].clip(upper=RUL_CAP)

# Training early fault label
train_df["early_fault"] = (train_df["RUL_raw"] <= FAULT_THRESHOLD).astype(int)

display(train_df[["unit", "cycle", "max_cycle", "RUL_raw", "RUL", "early_fault"]].head())
display(train_df[["unit", "cycle", "max_cycle", "RUL_raw", "RUL", "early_fault"]].tail())

print("Train RUL summary:")
print(train_df["RUL"].describe())

print("\nEarly fault distribution:")
print(train_df["early_fault"].value_counts())
print("Early fault ratio:", train_df["early_fault"].mean())

,unit,cycle,max_cycle,RUL_raw,RUL,early_fault
0,1,1,321,320,125,0
1,1,2,321,319,125,0
2,1,3,321,318,125,0
3,1,4,321,317,125,0
4,1,5,321,316,125,0


,unit,cycle,max_cycle,RUL_raw,RUL,early_fault
61244,249,251,255,4,4,1
61245,249,252,255,3,3,1
61246,249,253,255,2,2,1
61247,249,254,255,1,1,1
61248,249,255,255,0,0,1


Train RUL summary:
count    61249.000000
mean        92.985192
std         40.665112
min          0.000000
25%         61.000000
50%        122.000000
75%        125.000000
max        125.000000
Name: RUL, dtype: float64

Early fault distribution:
early_fault
0    53530
1     7719
Name: count, dtype: int64
Early fault ratio: 0.126026547372202


In [19]:
# Map each test unit to its final true RUL
test_rul_map = dict(zip(range(1, len(rul_df) + 1), rul_df["RUL"].values))

test_df["max_cycle"] = test_df.groupby("unit")["cycle"].transform("max")
test_df["final_RUL"] = test_df["unit"].map(test_rul_map)

# RUL at each test cycle
test_df["RUL_raw"] = test_df["final_RUL"] + (test_df["max_cycle"] - test_df["cycle"])
test_df["RUL"] = test_df["RUL_raw"].clip(upper=RUL_CAP)

# Test early fault label
test_df["early_fault"] = (test_df["RUL_raw"] <= FAULT_THRESHOLD).astype(int)

display(test_df[["unit", "cycle", "max_cycle", "final_RUL", "RUL_raw", "RUL", "early_fault"]].head())

,unit,cycle,max_cycle,final_RUL,RUL_raw,RUL,early_fault
0,1,1,230,22,251,125,0
1,1,2,230,22,250,125,0
2,1,3,230,22,249,125,0
3,1,4,230,22,248,125,0
4,1,5,230,22,247,125,0


In [20]:
from sklearn.model_selection import train_test_split

all_units = train_df["unit"].unique()

train_units, val_units = train_test_split(
    all_units,
    test_size=0.2,
    random_state=42
)

train_part = train_df[train_df["unit"].isin(train_units)].copy()
val_part   = train_df[train_df["unit"].isin(val_units)].copy()

print("Training engines:", len(train_units))
print("Validation engines:", len(val_units))
print("Train rows:", train_part.shape)
print("Val rows:", val_part.shape)

Training engines: 199
Validation engines: 50
Train rows: (49294, 30)
Val rows: (11955, 30)


In [21]:
"""
from sklearn.preprocessing import StandardScaler

# Drop near-zero variance sensors (confirmed flat in C-MAPSS)
DROP_SENSORS = ["s1", "s5", "s10", "s16", "s18", "s19"]
all_sensor_cols = [f"s{i}" for i in range(1, 22)]
feature_cols = ["setting_1", "setting_2", "setting_3"] + [s for s in all_sensor_cols if s not in DROP_SENSORS]

scaler = StandardScaler()
train_part[feature_cols] = scaler.fit_transform(train_part[feature_cols])
val_part[feature_cols] = scaler.transform(val_part[feature_cols])
test_scaled = test_df.copy()
test_scaled[feature_cols] = scaler.transform(test_scaled[feature_cols])

print("Number of input features:", len(feature_cols))  # should be 18
"""


'\nfrom sklearn.preprocessing import StandardScaler\n\n# Drop near-zero variance sensors (confirmed flat in C-MAPSS)\nDROP_SENSORS = ["s1", "s5", "s10", "s16", "s18", "s19"]\nall_sensor_cols = [f"s{i}" for i in range(1, 22)]\nfeature_cols = ["setting_1", "setting_2", "setting_3"] + [s for s in all_sensor_cols if s not in DROP_SENSORS]\n\nscaler = StandardScaler()\ntrain_part[feature_cols] = scaler.fit_transform(train_part[feature_cols])\nval_part[feature_cols] = scaler.transform(val_part[feature_cols])\ntest_scaled = test_df.copy()\ntest_scaled[feature_cols] = scaler.transform(test_scaled[feature_cols])\n\nprint("Number of input features:", len(feature_cols))  # should be 18\n'

In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

DROP_SENSORS = ["s1", "s5", "s10", "s16", "s18", "s19"]
all_sensor_cols = [f"s{i}" for i in range(1, 22)]
feature_cols = ["setting_1", "setting_2", "setting_3"] + [s for s in all_sensor_cols if s not in DROP_SENSORS]

# Detect operating conditions
N_CONDITIONS = train_part[["setting_1", "setting_2", "setting_3"]].nunique().max()
USE_CLUSTER_NORM = N_CONDITIONS > 2  # True for FD002, FD004

if USE_CLUSTER_NORM:
    print(f"Multi-condition dataset — using cluster-based normalization")
    N_CLUSTERS = 6

    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    train_part = train_part.copy()
    train_part["op_cluster"] = km.fit_predict(train_part[["setting_1", "setting_2", "setting_3"]])

    scaler_dict = {}
    for c in range(N_CLUSTERS):
        mask = train_part["op_cluster"] == c
        scaler_c = StandardScaler()
        train_part.loc[mask, feature_cols] = scaler_c.fit_transform(train_part.loc[mask, feature_cols])
        scaler_dict[c] = scaler_c

    def apply_cluster_scaler(df, km_model, scaler_dict, feature_cols):
        df = df.copy()
        df["op_cluster"] = km_model.predict(df[["setting_1", "setting_2", "setting_3"]])
        for c, scaler_c in scaler_dict.items():
            mask = df["op_cluster"] == c
            if mask.sum() > 0:
                df.loc[mask, feature_cols] = scaler_c.transform(df.loc[mask, feature_cols])
        return df

    val_part   = apply_cluster_scaler(val_part,   km, scaler_dict, feature_cols)
    test_scaled = apply_cluster_scaler(test_df.copy(), km, scaler_dict, feature_cols)

else:
    print(f"Single-condition dataset — using standard normalization")
    scaler = StandardScaler()
    train_part[feature_cols] = scaler.fit_transform(train_part[feature_cols])
    val_part[feature_cols]   = scaler.transform(val_part[feature_cols])
    test_scaled = test_df.copy()
    test_scaled[feature_cols] = scaler.transform(test_scaled[feature_cols])

print("Features:", len(feature_cols))

Multi-condition dataset — using cluster-based normalization


/tmp/ipykernel_14867/1767956019.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.27190928 -1.04409768 -1.65800348 ...  1.41152551  1.41152551
  2.0254313 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = scaler_c.fit_transform(train_part.loc[mask, feature_cols])
/tmp/ipykernel_14867/1767956019.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.79761971 -0.43019189  0.18371391 ... -0.43019189  0.79761971
  0.79761971]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = scaler_c.transform(df.loc[mask, feature_cols])


Features: 18


/tmp/ipykernel_14867/1767956019.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.43019189 -1.65800348 -1.65800348 ...  2.0254313   1.41152551
  0.79761971]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = scaler_c.transform(df.loc[mask, feature_cols])


In [23]:
WINDOW_SIZE = 30

def create_sliding_windows(df, feature_cols, window_size=30):
    X = []
    y_rul = []
    y_fault = []
    unit_ids = []
    end_cycles = []

    for unit in sorted(df["unit"].unique()):
        unit_df = df[df["unit"] == unit].sort_values("cycle").reset_index(drop=True)

        features = unit_df[feature_cols].values
        rul_values = unit_df["RUL"].values
        fault_values = unit_df["early_fault"].values
        cycles = unit_df["cycle"].values

        if len(unit_df) < window_size:
            continue

        for start in range(0, len(unit_df) - window_size + 1):
            end = start + window_size

            X.append(features[start:end])
            y_rul.append(rul_values[end - 1])
            y_fault.append(fault_values[end - 1])
            unit_ids.append(unit)
            end_cycles.append(cycles[end - 1])

    return (
        np.array(X, dtype=np.float32),
        np.array(y_rul, dtype=np.float32),
        np.array(y_fault, dtype=np.float32),
        np.array(unit_ids),
        np.array(end_cycles)
    )

X_train, y_rul_train, y_fault_train, train_window_units, train_end_cycles = create_sliding_windows(
    train_part, feature_cols, WINDOW_SIZE
)

X_val, y_rul_val, y_fault_val, val_window_units, val_end_cycles = create_sliding_windows(
    val_part, feature_cols, WINDOW_SIZE
)

X_test, y_rul_test, y_fault_test, test_window_units, test_end_cycles = create_sliding_windows(
    test_scaled, feature_cols, WINDOW_SIZE
)

print("X_train:", X_train.shape)
print("y_rul_train:", y_rul_train.shape)
print("y_fault_train:", y_fault_train.shape)

print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (43523, 30, 18)
y_rul_train: (43523,)
y_fault_train: (43523,)
X_val: (10505, 30, 18)
X_test: (34081, 30, 18)


In [24]:
y_rul_train_scaled = y_rul_train / RUL_CAP
y_rul_val_scaled   = y_rul_val / RUL_CAP
y_rul_test_scaled  = y_rul_test / RUL_CAP

print("Scaled RUL range:", y_rul_train_scaled.min(), y_rul_train_scaled.max())

Scaled RUL range: 0.0 1.0


In [25]:
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

input_shape = (WINDOW_SIZE, len(feature_cols))

inputs = layers.Input(shape=input_shape)

x = layers.Conv1D(filters=64, kernel_size=5, padding="same", activation="relu")(inputs)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Dropout(0.2)(x)

x = layers.Conv1D(filters=128, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Dropout(0.2)(x)

x = layers.Conv1D(filters=128, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)

x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)

rul_output = layers.Dense(1, activation="linear", name="rul_output")(x)
fault_output = layers.Dense(1, activation="sigmoid", name="fault_output")(x)

model = models.Model(inputs=inputs, outputs=[rul_output, fault_output])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "rul_output": "mse",
        "fault_output": "binary_crossentropy"
    },
    loss_weights={
        "rul_output": 1.0,
        "fault_output": 1.5
    },
    metrics={
        "rul_output": ["mae"],
        "fault_output": [
            "accuracy",
            tf.keras.metrics.AUC(curve="PR", name="auprc")
        ]
    }
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 30, 18)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 30, 64)    │      5,824 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 30, 64)    │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 15, 64)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 15, 64)    │          0 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 15, 128)   │     24,704 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 15, 128)   │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 7, 128)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 7, 128)    │          0 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 7, 128)    │     49,280 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 128)    │        512 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 7, 128)    │     98,816 │ batch_normalizat… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 7, 128)    │     33,088 │ bidirectional_1[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rul_output (Dense)  │ (None, 1)         │         65 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fault_output        │ (None, 1)         │         65 │ dropout_7[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 221,378 (864.76 KB)

 Trainable params: 220,738 (862.26 KB)

 Non-trainable params: 640 (2.50 KB)

In [26]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)
]

# Compute per-sample weights based on fault label
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_fault_train)
# Define weighted loss to replace sample_weight
def weighted_bce(pos_weight):
    def loss_fn(y_true, y_pred):
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        weight = y_true * pos_weight + (1 - y_true) * 1.0
        return tf.reduce_mean(weight * bce)
    return loss_fn

# Compute pos_weight from training data
neg = np.sum(y_fault_train == 0)
pos = np.sum(y_fault_train == 1)
pos_weight = neg / pos  # typically ~4-5 for FD001

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "rul_output": "mse",
        "fault_output": weighted_bce(pos_weight)   # ← replaces binary_crossentropy
    },
    loss_weights={
        "rul_output": 1.0,
        "fault_output": 1.5
    },
    metrics={
        "rul_output": ["mae"],
        "fault_output": [
            "accuracy",
            tf.keras.metrics.AUC(curve="PR", name="auprc")
        ]
    }
)

# Then fit normally — no sample_weight at all
history = model.fit(
    X_train,
    {"rul_output": y_rul_train_scaled, "fault_output": y_fault_train},
    validation_data=(
        X_val,
        {"rul_output": y_rul_val_scaled, "fault_output": y_fault_val}
    ),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
681/681 ━━━━━━━━━━━━━━━━━━━━ 51s 61ms/step - fault_output_accuracy: 0.9233 - fault_output_auprc: 0.8805 - fault_output_loss: 0.2879 - loss: 0.4821 - rul_output_loss: 0.0536 - rul_output_mae: 0.1756 - val_fault_output_accuracy: 0.9319 - val_fault_output_auprc: 0.8914 - val_fault_output_loss: 0.2597 - val_loss: 0.4385 - val_rul_output_loss: 0.0479 - val_rul_output_mae: 0.1858 - learning_rate: 0.0010
Epoch 2/50
681/681 ━━━━━━━━━━━━━━━━━━━━ 41s 60ms/step - fault_output_accuracy: 0.9401 - fault_output_auprc: 0.9095 - fault_output_loss: 0.2191 - loss: 0.3591 - rul_output_loss: 0.0308 - rul_output_mae: 0.1340 - val_fault_output_accuracy: 0.9313 - val_fault_output_auprc: 0.8982 - val_fault_output_loss: 0.2623 - val_loss: 0.4229 - val_rul_output_loss: 0.0279 - val_rul_output_mae: 0.1336 - learning_rate: 0.0010
Epoch 3/50
681/681 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - fault_output_accuracy: 0.9476 - fault_output_auprc: 0.9184 - fault_output_loss: 0.1898 - loss: 0.3116 - rul_output_loss:

In [27]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="train total loss")
plt.plot(history.history["val_loss"], label="val total loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history.history["rul_output_mae"], label="train RUL MAE scaled")
plt.plot(history.history["val_rul_output_mae"], label="val RUL MAE scaled")
plt.xlabel("Epoch")
plt.ylabel("Scaled MAE")
plt.title("RUL Prediction MAE")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history.history["fault_output_accuracy"], label="train fault accuracy")
plt.plot(history.history["val_fault_output_accuracy"], label="val fault accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Early Fault Detection Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [28]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

pred_rul_scaled, pred_fault_prob = model.predict(X_test)

pred_rul = pred_rul_scaled.flatten() * RUL_CAP
pred_fault_prob = pred_fault_prob.flatten()
pred_fault = (pred_fault_prob >= 0.5).astype(int)

mae = mean_absolute_error(y_rul_test, pred_rul)
rmse = np.sqrt(mean_squared_error(y_rul_test, pred_rul))
r2 = r2_score(y_rul_test, pred_rul)

acc = accuracy_score(y_fault_test, pred_fault)
prec = precision_score(y_fault_test, pred_fault, zero_division=0)
rec = recall_score(y_fault_test, pred_fault, zero_division=0)
f1 = f1_score(y_fault_test, pred_fault, zero_division=0)
auprc = average_precision_score(y_fault_test, pred_fault_prob)

try:
    auroc = roc_auc_score(y_fault_test, pred_fault_prob)
except:
    auroc = None

print("===== Test Results: All Test Windows =====")
print(f"RUL MAE:  {mae:.4f}")
print(f"RUL RMSE: {rmse:.4f}")
print(f"RUL R2:   {r2:.4f}")

print(f"Fault Accuracy:  {acc:.4f}")
print(f"Fault Precision: {prec:.4f}")
print(f"Fault Recall:    {rec:.4f}")
print(f"Fault F1:        {f1:.4f}")
print(f"Fault AUPRC:     {auprc:.4f}")
print(f"Fault AUROC:     {auroc}")

1066/1066 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step
===== Test Results: All Test Windows =====
RUL MAE:  19.6163
RUL RMSE: 22.6320
RUL R2:   0.3134
Fault Accuracy:  0.9742
Fault Precision: 0.4953
Fault Recall:    0.9109
Fault F1:        0.6417
Fault AUPRC:     0.7419
Fault AUROC:     0.9919735501901638


In [29]:
test_window_info = pd.DataFrame({
    "idx": np.arange(len(test_window_units)),
    "unit": test_window_units,
    "end_cycle": test_end_cycles
})

last_window_indices = (
    test_window_info
    .sort_values("end_cycle")
    .groupby("unit")["idx"]
    .last()
    .values
)

X_test_last = X_test[last_window_indices]
y_rul_test_last = y_rul_test[last_window_indices]
y_fault_test_last = y_fault_test[last_window_indices]

pred_rul_scaled_last, pred_fault_prob_last = model.predict(X_test_last)

pred_rul_last = pred_rul_scaled_last.flatten() * RUL_CAP
pred_fault_prob_last = pred_fault_prob_last.flatten()
pred_fault_last = (pred_fault_prob_last >= 0.5).astype(int)

mae_last = mean_absolute_error(y_rul_test_last, pred_rul_last)
rmse_last = np.sqrt(mean_squared_error(y_rul_test_last, pred_rul_last))
r2_last = r2_score(y_rul_test_last, pred_rul_last)

acc_last = accuracy_score(y_fault_test_last, pred_fault_last)
prec_last = precision_score(y_fault_test_last, pred_fault_last, zero_division=0)
rec_last = recall_score(y_fault_test_last, pred_fault_last, zero_division=0)
f1_last = f1_score(y_fault_test_last, pred_fault_last, zero_division=0)
auprc_last = average_precision_score(y_fault_test_last, pred_fault_prob_last)

try:
    auroc_last = roc_auc_score(y_fault_test_last, pred_fault_prob_last)
except:
    auroc_last = None

print("===== Test Results: Final Window Per Test Engine =====")
print(f"RUL MAE:  {mae_last:.4f}")
print(f"RUL RMSE: {rmse_last:.4f}")
print(f"RUL R2:   {r2_last:.4f}")

print(f"Fault Accuracy:  {acc_last:.4f}")
print(f"Fault Precision: {prec_last:.4f}")
print(f"Fault Recall:    {rec_last:.4f}")
print(f"Fault F1:        {f1_last:.4f}")
print(f"Fault AUPRC:     {auprc_last:.4f}")
print(f"Fault AUROC:     {auroc_last}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
===== Test Results: Final Window Per Test Engine =====
RUL MAE:  16.8138
RUL RMSE: 20.7194
RUL R2:   0.7651
Fault Accuracy:  0.9198
Fault Precision: 0.7361
Fault Recall:    1.0000
Fault F1:        0.8480
Fault AUPRC:     0.9559
Fault AUROC:     0.9865668580803938


Save all

In [30]:
import os
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_recall_curve

# ── folder setup ──────────────────────────────────────────────────────────────
BASE_RESULT = "/content/drive/MyDrive/TASK-ERAU/results"
MODEL_NAME  = "1D-CNN-BiLSTM-MultiHeadAttention-DualOutputHeads"
FD_NAME     = "FD004"
SAVE_DIR    = os.path.join(BASE_RESULT, MODEL_NAME, FD_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 1. training curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history.history["loss"], label="train"); axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Total loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history["rul_output_mae"], label="train"); axes[1].plot(history.history["val_rul_output_mae"], label="val")
axes[1].set_title("RUL MAE (scaled)"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(history.history["fault_output_accuracy"], label="train"); axes[2].plot(history.history["val_fault_output_accuracy"], label="val")
axes[2].set_title("Fault accuracy"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── 2. all test windows evaluation ───────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score, roc_auc_score

pred_rul_scaled, pred_fault_prob = model.predict(X_test)
pred_rul = pred_rul_scaled.flatten() * RUL_CAP
pred_fault_prob = pred_fault_prob.flatten()
pred_fault = (pred_fault_prob >= 0.5).astype(int)

mae  = mean_absolute_error(y_rul_test, pred_rul)
rmse = np.sqrt(mean_squared_error(y_rul_test, pred_rul))
r2   = r2_score(y_rul_test, pred_rul)
acc  = accuracy_score(y_fault_test, pred_fault)
prec = precision_score(y_fault_test, pred_fault, zero_division=0)
rec  = recall_score(y_fault_test, pred_fault, zero_division=0)
f1   = f1_score(y_fault_test, pred_fault, zero_division=0)
auprc = average_precision_score(y_fault_test, pred_fault_prob)
try:    auroc = roc_auc_score(y_fault_test, pred_fault_prob)
except: auroc = None

print("===== Test Results: All Test Windows =====")
print(f"RUL MAE:  {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")
print(f"Fault Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")
print(f"AUPRC: {auprc:.4f} | AUROC: {auroc:.4f}")

# ── 3. last window per engine evaluation ──────────────────────────────────────
test_window_info = pd.DataFrame({
    "idx": np.arange(len(test_window_units)),
    "unit": test_window_units,
    "end_cycle": test_end_cycles
})
last_window_indices = (
    test_window_info.sort_values("end_cycle")
    .groupby("unit")["idx"].last().values
)

X_test_last      = X_test[last_window_indices]
y_rul_test_last  = y_rul_test[last_window_indices]
y_fault_test_last = y_fault_test[last_window_indices]

pred_rul_scaled_last, pred_fault_prob_last = model.predict(X_test_last)
pred_rul_last       = pred_rul_scaled_last.flatten() * RUL_CAP
pred_fault_prob_last = pred_fault_prob_last.flatten()
pred_fault_last     = (pred_fault_prob_last >= 0.5).astype(int)

mae_last  = mean_absolute_error(y_rul_test_last, pred_rul_last)
rmse_last = np.sqrt(mean_squared_error(y_rul_test_last, pred_rul_last))
r2_last   = r2_score(y_rul_test_last, pred_rul_last)
acc_last  = accuracy_score(y_fault_test_last, pred_fault_last)
prec_last = precision_score(y_fault_test_last, pred_fault_last, zero_division=0)
rec_last  = recall_score(y_fault_test_last, pred_fault_last, zero_division=0)
f1_last   = f1_score(y_fault_test_last, pred_fault_last, zero_division=0)
auprc_last = average_precision_score(y_fault_test_last, pred_fault_prob_last)
try:    auroc_last = roc_auc_score(y_fault_test_last, pred_fault_prob_last)
except: auroc_last = None

print("\n===== Test Results: Final Window Per Test Engine =====")
print(f"RUL MAE:  {mae_last:.4f} | RMSE: {rmse_last:.4f} | R2: {r2_last:.4f}")
print(f"Fault Acc: {acc_last:.4f} | Prec: {prec_last:.4f} | Rec: {rec_last:.4f} | F1: {f1_last:.4f}")
print(f"AUPRC: {auprc_last:.4f} | AUROC: {auroc_last:.4f}")

# ── 4. RUL scatter ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_rul_test_last, pred_rul_last, alpha=0.7, edgecolors="k", linewidths=0.4)
lim = max(y_rul_test_last.max(), pred_rul_last.max()) + 5
ax.plot([0, lim], [0, lim], "r--", label="Perfect prediction")
ax.set_xlabel("Actual RUL"); ax.set_ylabel("Predicted RUL")
ax.set_title(f"RUL: Predicted vs Actual\nMAE={mae_last:.2f}  RMSE={rmse_last:.2f}  R²={r2_last:.3f}")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "rul_scatter.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── 5. confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_fault_test_last, pred_fault_last)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["No fault", "Fault"]).plot(ax=ax, colorbar=False)
ax.set_title("Fault detection — confusion matrix (last window)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fault_confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── 6. PR curve ───────────────────────────────────────────────────────────────
precision_curve, recall_curve, _ = precision_recall_curve(y_fault_test_last, pred_fault_prob_last)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall_curve, precision_curve, lw=2)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall curve  |  AUPRC={auprc_last:.4f}")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fault_pr_curve.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── 7. metrics txt ────────────────────────────────────────────────────────────
metrics_text = f"""===== {MODEL_NAME} | {FD_NAME} =====

-- All test windows --
RUL MAE:         {mae:.4f}
RUL RMSE:        {rmse:.4f}
RUL R2:          {r2:.4f}
Fault Accuracy:  {acc:.4f}
Fault Precision: {prec:.4f}
Fault Recall:    {rec:.4f}
Fault F1:        {f1:.4f}
Fault AUPRC:     {auprc:.4f}
Fault AUROC:     {auroc:.4f}

-- Last window per engine --
RUL MAE:         {mae_last:.4f}
RUL RMSE:        {rmse_last:.4f}
RUL R2:          {r2_last:.4f}
Fault Accuracy:  {acc_last:.4f}
Fault Precision: {prec_last:.4f}
Fault Recall:    {rec_last:.4f}
Fault F1:        {f1_last:.4f}
Fault AUPRC:     {auprc_last:.4f}
Fault AUROC:     {auroc_last:.4f}
"""
with open(os.path.join(SAVE_DIR, "metrics.txt"), "w") as f:
    f.write(metrics_text)

print(f"\n✓ All results saved to: {SAVE_DIR}")

1066/1066 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step
===== Test Results: All Test Windows =====
RUL MAE:  19.6163 | RMSE: 22.6320 | R2: 0.3134
Fault Acc: 0.9742 | Prec: 0.4953 | Rec: 0.9109 | F1: 0.6417
AUPRC: 0.7419 | AUROC: 0.9920
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 

===== Test Results: Final Window Per Test Engine =====
RUL MAE:  16.8138 | RMSE: 20.7194 | R2: 0.7651
Fault Acc: 0.9198 | Prec: 0.7361 | Rec: 1.0000 | F1: 0.8480
AUPRC: 0.9559 | AUROC: 0.9866

✓ All results saved to: /content/drive/MyDrive/TASK-ERAU/results/1D-CNN-BiLSTM-MultiHeadAttention-DualOutputHeads/FD004
